# Assembling a Crew

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app).

You will build a small CrewAI crew with two agents and two tasks, run it with `kickoff(inputs=...)`, inspect `CrewOutput`, and batch runs with `kickoff_for_each`.

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Basic crew: two agents, two tasks, `kickoff` with inputs

Placeholders like `{topic}` in task descriptions are filled from the `inputs` dict passed to `kickoff`.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Find accurate, concise facts on the given topic",
    backstory="You summarize clearly and avoid speculation.",
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Turn research into clear prose",
    backstory="You write tight, readable paragraphs.",
    verbose=True,
)

research_task = Task(
    description="Research the topic: {topic}. List 3–5 bullet facts.",
    expected_output="A short bullet list of facts.",
    agent=researcher,
)

write_task = Task(
    description="Using the research, write one paragraph about {topic}.",
    expected_output="One paragraph of prose.",
    agent=writer,
    context=[research_task],
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff(inputs={"topic": "renewable energy"})

## Inspecting `CrewOutput`

The object returned by `kickoff` exposes `.raw`, per-task `.tasks_output`, `.token_usage`, and (when you configure structured outputs on tasks) `.pydantic` and `.json_dict`.

In [ ]:
print("raw:\n", result.raw)
print("tasks_output:", result.tasks_output)
print("token_usage:", result.token_usage)
print("pydantic:", result.pydantic)
print("json_dict:", result.json_dict)

## Batch runs with `kickoff_for_each`

Run the same crew once per input dictionary.

In [ ]:
batch_inputs = [
    {"topic": "machine learning"},
    {"topic": "supply chains"},
]

batch_results = crew.kickoff_for_each(inputs=batch_inputs)

for i, out in enumerate(batch_results):
    print(f"--- run {i} ---")
    print(out.raw[:500], "..." if len(out.raw) > 500 else "")

## Key takeaways

- A **Crew** wires **agents** to **tasks** and a **process** (`Process.sequential` is the default).
- Use **`kickoff(inputs={...})`** so `{placeholders}` in task text are filled at run time.
- Read **`CrewOutput`** via **`.raw`**, **`.tasks_output`**, and **`.token_usage`**; **`.pydantic`** / **`.json_dict`** matter when tasks use structured outputs.
- Use **`kickoff_for_each`** to reuse one crew definition across many input dicts.
- For larger apps, consider the **`@CrewBase`** pattern with **`@agent`**, **`@task`**, **`@crew`**, **`@before_kickoff`**, and **`@after_kickoff`**.